In [1]:
import pandas as pd

DRI_1 = "results/DRI/decarbonization_readiness_index.parquet"
DRI_2 = "results/DRI/decarbonization_readiness_index_scope_12_only.parquet"

# Load
dri_a = pd.read_parquet(DRI_1)
dri_b = pd.read_parquet(DRI_2)

# Basic hygiene
for df in (dri_a, dri_b):
    df.columns = df.columns.str.strip()
    df["Sector"] = df["Sector"].astype(str).str.strip()

# Rename DRI columns to avoid collision
dri_a = dri_a.rename(columns={"DRI": "DRI_A"})
dri_b = dri_b.rename(columns={"DRI": "DRI_B"})

# Merge on Sector
merged = dri_a[["Sector", "DRI_A"]].merge(
    dri_b[["Sector", "DRI_B"]],
    on="Sector",
    how="inner"
)

print("Merged sectors:", len(merged))

# Drop NaNs just in case
merged = merged.dropna(subset=["DRI_A", "DRI_B"])

# Correlations
pearson  = merged["DRI_A"].corr(merged["DRI_B"], method="pearson")
spearman = merged["DRI_A"].corr(merged["DRI_B"], method="spearman")

print(f"DRI(A) vs DRI(B)")
print(f"  Pearson : {pearson:.4f}")
print(f"  Spearman: {spearman:.4f}")


Merged sectors: 11
DRI(A) vs DRI(B)
  Pearson : 0.2272
  Spearman: 0.2727


In [2]:
dri_a.columns

Index(['Sector', 'Room_norm', 'Flex_norm', 'Sens_norm', 'Robust_norm',
       'DRI_A'],
      dtype='object')

In [3]:
dims = [ 'Room_norm', 'Flex_norm', 'Sens_norm', 'Robust_norm']  # adapt to your actual names

# Rename to avoid collisions
dri_a = dri_a.rename(columns={d: f"{d}_A" for d in dims})
dri_b = dri_b.rename(columns={d: f"{d}_B" for d in dims})

# Merge
merged = dri_a[["Sector"] + [f"{d}_A" for d in dims]].merge(
    dri_b[["Sector"] + [f"{d}_B" for d in dims]],
    on="Sector",
    how="inner"
)

merged = merged.dropna()
print("Merged sectors:", len(merged))


Merged sectors: 11


In [4]:
rows = []

pretty_names = {
    "Room_norm":   "Room for Maneuver",
    "Flex_norm":   "Flexibility",
    "Sens_norm":   "Sensitivity",
    "Robust_norm": "Robustness"
}

for d in dims:
    pearson  = merged[f"{d}_A"].corr(merged[f"{d}_B"], method="pearson")
    spearman = merged[f"{d}_A"].corr(merged[f"{d}_B"], method="spearman")

    rows.append({
        "Dimension": pretty_names.get(d, d),
        "Pearson": round(pearson, 3),
        "Spearman": round(spearman, 3)
    })

corr_df = pd.DataFrame(rows)

latex_table = corr_df.to_latex(
    index=False,
    column_format="lcc",
    caption="Correlation between dimension scores computed using Scope~1--3 emissions and their counterparts based on Scope~1--2 emissions.",
    label="tab:scope3_correlation",
    escape=False
)

print(latex_table)



\begin{table}
\caption{Correlation between dimension scores computed using Scope~1--3 emissions and their counterparts based on Scope~1--2 emissions.}
\label{tab:scope3_correlation}
\begin{tabular}{lcc}
\toprule
Dimension & Pearson & Spearman \\
\midrule
Room for Maneuver & 0.411000 & 0.382000 \\
Flexibility & 0.228000 & 0.391000 \\
Sensitivity & 0.869000 & 0.818000 \\
Robustness & 0.956000 & 0.882000 \\
\bottomrule
\end{tabular}
\end{table}

